# Polymarket Research — Data Exploration

Starter notebook for analyzing collected prediction market data.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import sqlite3
import pandas as pd
import config

conn = sqlite3.connect(config.DB_PATH)
print(f"Connected to: {config.DB_PATH}")

## Database Overview

In [ ]:
for table in ['events', 'markets', 'trades', 'wallet_profiles']:
    count = pd.read_sql(f"SELECT COUNT(*) as n FROM {table}", conn).iloc[0]['n']
    print(f"{table:<20} {count:>8,} rows")

## Top Whales by Weather PnL

In [ ]:
whales = pd.read_sql("""
    SELECT proxy_wallet, weather_pnl, total_pnl, win_rate, sharpe_ratio,
           edge_decay, sybil_risk_flag
    FROM wallet_profiles
    WHERE weather_pnl IS NOT NULL AND weather_pnl > 0
    ORDER BY weather_pnl DESC
    LIMIT 20
""", conn)

whales['win_rate_pct'] = (whales['win_rate'] * 100).round(1)
whales

## Markets with Price < 5% (Tail Harvesting Candidates)

In [ ]:
tails = pd.read_sql("""
    SELECT m.slug, m.event_slug, m.volume_total, m.closed,
           MIN(t.price) as min_price,
           AVG(t.price) as avg_price,
           COUNT(t.id) as trade_count
    FROM markets m
    JOIN trades t ON t.slug = m.slug
    WHERE t.price > 0 AND t.price < 0.05
    GROUP BY m.slug
    HAVING trade_count > 5
    ORDER BY m.volume_total DESC
    LIMIT 20
""", conn)

tails['avg_price_pct'] = (tails['avg_price'] * 100).round(2)
tails

## Trade Count by Market (Top 20)

In [ ]:
trade_counts = pd.read_sql("""
    SELECT slug, COUNT(*) as trade_count, 
           COUNT(DISTINCT proxy_wallet) as unique_traders,
           MIN(price) as min_price,
           MAX(price) as max_price,
           AVG(price) as avg_price
    FROM trades
    GROUP BY slug
    ORDER BY trade_count DESC
    LIMIT 20
""", conn)

trade_counts

## Events with Most Markets

In [ ]:
event_sizes = pd.read_sql("""
    SELECT e.event_slug, e.title, e.total_volume,
           COUNT(m.condition_id) as market_count
    FROM events e
    LEFT JOIN markets m ON m.event_slug = e.event_slug
    GROUP BY e.event_slug
    ORDER BY market_count DESC
    LIMIT 15
""", conn)

event_sizes

In [ ]:
conn.close()
print("Done.")